# 07 — Controlled simulation validation

Simulation provides known innovation regimes. This smoke design compares the classical models under all four regimes. Set `RUN_DIFFUSION=True` only after the financial DI-VAR notebook succeeds. Final research results should repeat the experiment across many seeds.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from innovcal.data.simulation import generate_multiple_var_datasets, make_stable_var_matrix
from innovcal.experiments import run_financial_experiment

In [2]:
K = 4
A = make_stable_var_matrix(K, scale=0.4, seed=10)
Sigma = 0.2 * np.ones((K, K)) + 0.8 * np.eye(K)
datasets = generate_multiple_var_datasets(
    ['gaussian', 'student_t', 'mixture', 'heteroskedastic'],
    n_obs=500, burn_in=100, A=A, Sigma=Sigma, base_seed=200,
)
RUN_DIFFUSION = True
methods = ['gaussian', 'student_t', 'bootstrap'] + (['diffusion'] if RUN_DIFFUSION else [])

/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:47: RuntimeWarning: divide by zero encountered in matmul
  return rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:47: RuntimeWarning: overflow encountered in matmul
  return rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:47: RuntimeWarning: invalid value encountered in matmul
  return rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:66: RuntimeWarning: divide by zero encountered in matmul
  z = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:66: RuntimeWarning: overflow encountered in matmul
  z = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/data/simulation.py:66: RuntimeWarning: invalid value encountered in

In [3]:
tables = []
for name, dataset in datasets.items():
    result = run_financial_experiment(
        dataset['y'], methods=tuple(methods), lags=1, n_paths=200, seed=123,
        diffusion_options={
            'timesteps': 50, 'epochs': 50, 'hidden_dim': 64,
            'time_embedding_dim': 16, 'verbose': False,
        },
    )
    table = result.evaluation.copy()
    table['dgp'] = name
    tables.append(table)
simulation_results = pd.concat(tables, ignore_index=True)
simulation_results.sort_values(['dgp', 'energy_score'])

/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: divide by zero encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: overflow encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: invalid value encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: divide by zero encountered in matmul
  shocks = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: overflow encountered in matmul
  shocks = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: invalid value encountered in matmul
  shocks = rng.

,dgp,forecast_model,innovation_model,avg_coverage,avg_width,energy_score,crps,interval_score,ece,pit_deviation,...,width_1,coverage_2,width_2,coverage_3,width_3,coverage_4,width_4,nominal_coverage,coverage_error,abs_coverage_error
0,gaussian,VAR,gaussian,0.8750,3.407228,1.463816,0.623407,4.503663,0.024167,0.0175,...,3.239886,0.88,3.184744,0.91,3.610802,0.80,3.593481,0.9,-2.500000e-02,2.500000e-02
1,gaussian,VAR,student_t,0.8350,3.251272,1.473945,0.628739,4.583418,0.078333,0.0220,...,3.109904,0.81,3.052489,0.88,3.425879,0.77,3.416815,0.9,-6.500000e-02,6.500000e-02
3,gaussian,VAR,diffusion,0.9125,3.958916,1.473966,0.627428,4.535867,0.025833,0.0210,...,3.778295,0.90,3.592965,0.96,4.167034,0.85,4.297371,0.9,1.250000e-02,1.250000e-02
2,gaussian,VAR,bootstrap,0.8675,3.313127,1.478304,0.628194,4.643724,0.040000,0.0185,...,3.168936,0.86,3.221550,0.90,3.429428,0.79,3.432594,0.9,-3.250000e-02,3.250000e-02
14,heteroskedastic,VAR,bootstrap,0.9000,6.067577,2.026326,0.860724,8.960995,0.037500,0.0230,...,6.593451,0.88,6.133684,0.89,5.827334,0.93,5.715839,0.9,1.110223e-16,1.110223e-16
13,heteroskedastic,VAR,student_t,0.8825,5.566857,2.099708,0.896157,8.874724,0.078333,0.0515,...,6.148288,0.87,5.550965,0.89,5.261356,0.90,5.306819,0.9,-1.750000e-02,1.750000e-02
12,heteroskedastic,VAR,gaussian,0.8925,5.818448,2.157880,0.921844,9.002570,0.092500,0.0555,...,6.423301,0.89,5.825973,0.87,5.532601,0.92,5.491918,0.9,-7.500000e-03,7.500000e-03
15,heteroskedastic,VAR,diffusion,0.9125,6.808390,2.211770,0.941514,9.033997,0.110833,0.0605,...,7.426609,0.92,6.943647,0.92,6.625527,0.93,6.237777,0.9,1.250000e-02,1.250000e-02
10,mixture,VAR,bootstrap,0.9350,5.437981,1.769313,0.748382,6.690732,0.054167,0.0185,...,4.572048,0.91,4.750102,0.92,5.822479,0.98,6.607296,0.9,3.500000e-02,3.500000e-02
9,mixture,VAR,student_t,0.9450,5.625754,1.778961,0.755932,6.715844,0.086667,0.0290,...,4.708327,0.93,4.924704,0.95,6.352931,0.98,6.517056,0.9,4.500000e-02,4.500000e-02


In [4]:
CACHE = ROOT / 'results/notebook_cache'
simulation_results.to_csv(CACHE / 'simulation_results.csv', index=False)